The turtle command **goto**($x,y$) draws a straight line to ($x,y$) from the current position **pos()**. Suppose we move in $n$ equal steps from (0,0) to ($xmax,ymax$), with horizontal nudge $dx=xmax/n$ and vertical nudge $dy=ymax/n$. The code below does this and draws a right triangle underneath. Since $dy/dx$ is constant, the picture looks the same regardless of $n$, which the code renames as "steps".

In [1]:
from turtle import *   # import turtle commands
xmax,ymax,steps = 200,150,10   # max x position, max y position, and number of steps
dx,dy = xmax/steps, ymax/steps
reset()                          # reset screen and start at (0,0)
color('black','light yellow')
begin_fill()
xnew = ynew = 0
for i in range(steps):
    xnew += dx
    ynew += dy
    goto(xnew, ynew)
goto(xnew,0)          # draw vertical line back to starting y value of 0
home()                # draw horizontal line back to start at (0,0)
end_fill()

To fit stairs $dx$ wide just *outside* the triangle, turtle at ($x,y$) moves $dy$ vertically and then $dx$ horizontally. The area under that step will equal ($y+dx$)$*dx$.

In [2]:
color('tan1','tan1')
begin_fill()
xnew = ynew = area = 0
for i in range(steps):
    ynew += dy       # ynew nudges up just before the area is computed
    area += ynew * dx   
    goto(xnew, ynew)
    xnew += dx
    goto(xnew, ynew)
goto(xnew,0)
home()
end_fill()
color('black')
goto(xmax,ymax)
home()
print('Total area is',area)

Total area is 16500.0


To fit stairs $dx$ wide just *inside* the triangle, turtle at ($x,y$) moves $dx$ horizontally and then $dy$ vertically. The area under that step will equal $y*dx$.

In [3]:
color('tan4','tan4')
begin_fill()
xnew = ynew = area = 0
for i in range(steps):
    xnew += dx
    area += ynew * dx   
    goto(xnew, ynew)
    ynew += dy       # ynew doesn't increase until after area computed
    goto(xnew, ynew)
goto(xnew,0)
home()   
end_fill()
print('Total area is',area)

Total area is 13500.0


Let's store coordinates of the $x$ and $y$ positions along the way. They can be computed in several ways.

In [4]:
# long way to form x list
x = []
for i in range(steps+1):
    x.append(i * dx)
print('x is',x)

# short way to form x using list comprehension
xx = [i * dx for i in range(steps+1)]
print('Are x and xx the same list?',x==xx)

# two ways to form y 
y = [i * dy for i in range(steps+1)]  # will only work if we know dy before we know y
print('y is',y)
yy = [val * (ymax/xmax) for val in x]  # defines y as constant multiple of x
print('Are y and yy the same list?',y==yy)

x is [0.0, 20.0, 40.0, 60.0, 80.0, 100.0, 120.0, 140.0, 160.0, 180.0, 200.0]
Are x and xx the same list? True
y is [0.0, 15.0, 30.0, 45.0, 60.0, 75.0, 90.0, 105.0, 120.0, 135.0, 150.0]
Are y and yy the same list? True


Indices makes clearer what is already evident in the graphs, namely that the higher area includes a rectangle of height $ymax$ that the lower area does not. 

In [5]:
# this code presumes x[0]=y[0]=0
xmax,ymax,steps = 200,150,10
dx = xmax / steps
x = [dx * i  for i in range(steps+1)]
y = [(ymax/xmax) * val for val in x]  # defines y as constant multiple of x
# plot lower staircase and compute area
area_low = 0
for i in range(steps):
    goto(x[i+1], y[i])
    goto(x[i+1], y[i+1])
    area_low += y[i] * dx
home()
# plot upper staircasea and compute area
area_high = 0
for i in range(steps):
    goto(x[i], y[i+1])
    goto(x[i+1], y[i+1])
    area_high += y[i+1] * dx
home()
print(round(area_low),'< area of triangle <',round(area_high))

13500 < area of triangle < 16500


Here's a simpler way to compute areas using **sum( )**. The relative difference between the high and low areas works out to 2/steps. So we would need 200 steps to bound the areas to 1% accuracy.

In [6]:
area_low = sum(y[:-1]) * dx        # sum omits y[-1], which equals ymax 
area_high = sum(y[1:]) * dx        # sum omits y[0], which equals minimum y 
diff_area = area_high - area_low   # = ymax * dx = ymax * xmax/steps
area_mid =  0.5 * (area_low + area_high)
rel_diff = diff_area / area_mid    # = 2/steps 
print('Does the relative difference',rel_diff,' in areas equal 2/steps?',abs(rel_diff-2/steps)<1E-10)      

Does the relative difference 0.2  in areas equal 2/steps? True


We need a huge number of steps to make the relative difference insignificant. That's why it's better to use precise formulas if we can find them.

In [7]:
xmax,ymax,steps = 200,150,10
while steps < 1E6:
    steps *= 10
    dx = xmax / steps
    x = [i * dx for i in range(steps+1)]
    y = [(ymax/xmax) * val for val in x]
    area_low = sum(y[:-1]) * dx  
    area_high = sum(y[1:]) * dx
    area_mid = 0.5 * (area_low + area_high)
    rel_diff = (area_high - area_low) / area_mid 
    print(steps,'steps with 2/steps',2/steps,'and relative difference',rel_diff)

100 steps with 2/steps 0.02 and relative difference 0.02
1000 steps with 2/steps 0.002 and relative difference 0.002
10000 steps with 2/steps 0.0002 and relative difference 0.0002
100000 steps with 2/steps 2e-05 and relative difference 1.9999999999951494e-05
1000000 steps with 2/steps 2e-06 and relative difference 1.9999999999223914e-06


Now let's apply the same approach to to bound the area of a quarter circle. We will need to compute square roots, which requires a command **sqrt()** from the package math. We will type "import math"and "math.sqrt()" to use it. Why not type "from math import $*$"? That will overwrite at least one built-in command with a math package variant. In general it's safer not to make wildcard imports, although we made an exception for turtle to keep things simple.

In [8]:
import math  # this contains the command sqrt() and a host of other mathematical functions
# sqrt(2)    # this will generate an error, because Python doesn't see sqrt in the main list
root2 = math.sqrt(2)  # Python looks in the loaded math package for sqrt, finds it, and applies it
print('The square root of 2 is',root2)
print('The stored value of π is',math.pi) # math.pi is float approximation to π 

The square root of 2 is 1.4142135623730951
The stored value of π is 3.141592653589793


A circle around the origin with radius $r$ is defined by points $(x,y)$ satisfying $x^2+y^2=r^2$. We will focus on the quarter circle with negative $x$ and positive $y$. We will set each $x$ an equal distance $dx$ apart and approximate the quarter circle with polygons through the selected $(x,y)$ and the origin. $x \cdot x+y \cdot y=r \cdot r$. 

In [9]:
# calculate circle coordinates an equal dx apart, with x negative and y positive
radius,steps = 200,40
reset()
dx = radius/steps
x = [-radius + i * dx for i in range(steps+1)]
y = [math.sqrt(radius * radius - val * val) for val in x]
# draw a polygon with those points and the origin as vertices
color('blue','cyan')
begin_fill()
for i in range(steps+1):
    goto(x[i],y[i])
home()
end_fill()

Next let's build staircases just inside and outside the quarter circle. Their areas bound the $\frac{1}{4}\pi r^2$ area of the quarter circle, and we can use this to bound $\pi$.

In [10]:
# nest staircases inside and outside quarter circle and use areas to bound π 
color('black')
goto(x[0],0)
for i in range(steps): # upper steps
    goto(x[i],y[i+1])
    goto(x[i+1],y[i+1])
home()
color('light pink','light pink')
begin_fill()
goto(x[0],0)
for i in range(steps): # lower steps
    goto(x[i+1],y[i])
    goto(x[i+1],y[i+1])
home()
end_fill()
# compute areas and convert to bounds on π
area_high = sum(y[1:]) * dx 
area_low = sum(y[:-1]) * dx 
pi_high = area_high * 4 / (radius * radius)
pi_low = area_low * 4 / (radius * radius)
print(pi_low,'< π <',pi_high)

3.0869477342170106 < π < 3.1869477342170107


Those bounds are much looser than we computed in the notebook Ch2_Sec3. While every quadrupling of steps improves them, convergence is slow, as we see below.

In [11]:
# repeatedly quadruple steps to tighten bounds on π 
radius,steps = 200,40
for iter in range(6):
    steps *= 4
    dx = radius/steps
    x = [-radius + i * dx  for i in range(steps+1)]
    y = [math.sqrt(radius * radius - val * val) for val in x]
    area_high = sum(y[1:]) * dx  
    area_low = sum(y[:-1]) * dx  
    pi_high = area_high * 4 / (radius * radius)
    pi_low = area_low * 4 / (radius * radius)
    rel_diff = (pi_high - pi_low) / math.pi
    # any terms with parentheses around them can be extended to multiple lines
    print(steps,'steps:',round(pi_low,7),'< π <',round(pi_high,7),   # continues on next line
                             'with relative difference',round(rel_diff,6))

160 steps: 3.1285117 < π < 3.1535117 with relative difference 0.007958
640 steps: 3.138395 < π < 3.144645 with relative difference 0.001989
2560 steps: 3.1408023 < π < 3.1423648 with relative difference 0.000497
10240 steps: 3.1413962 < π < 3.1417868 with relative difference 0.000124
40960 steps: 3.1415437 < π < 3.1416413 with relative difference 3.1e-05
163840 steps: 3.1415804 < π < 3.1416048 with relative difference 8e-06


Our estimated bounds on π in Ch2_Sec3 relied on turtle's turns and distance estimators. Let's compute lower bounds on π from the sums of distances between the vertices on the rim of the quarter circle. Again, they are much lower than the corresponding area-based estimates. 

In [15]:
# estimate π by comparing outer perimeter to radius for approximate quarter circle
radius,steps = 200,40
pi_est = []
print('steps    estimated π      too low by')
for iter in range(6):
    steps *= 4
    dx = radius/steps
    x = [-radius + i * dx  for i in range(steps+1)]
    y = [math.sqrt(radius * radius - val * val) for val in x]
    quarterperim = 0
    for i in range(steps):
        dy = y[i+1] - y[i]
        quarterperim += math.sqrt(dx*dx + dy*dy)
    pi_low = 2 * quarterperim / radius
    diff_pi = math.pi - pi_low
    print(steps,'   ',round(pi_low,10),'    ', round(diff_pi,9))
    pi_est.append([steps, pi_low, diff_pi])
print('These answers stored as list of lists')
print(pi_est)    

steps    estimated π      too low by
160     3.1414473607      0.000145293
640     3.1415744946      1.8159e-05
2560     3.1415903838      2.27e-06
10240     3.1415923699      2.84e-07
40960     3.1415926181      3.5e-08
163840     3.1415926492      4e-09
These answers stored as list of lists
[[160, 3.1414473606927915, 0.0001452928970016032], [640, 3.1415744946045527, 1.815898524037962e-05], [2560, 3.1415903837982984, 2.2697914947578113e-06], [10240, 3.141592369868404, 2.8372138904941835e-07], [40960, 3.1415926181246916, 3.5465101522191844e-08], [163840, 3.1415926491566366, 4.433156508554248e-09]]


Before closing this lesson, let's discuss a potential problem in working with lists. Suppose we want to make a backup copy of list $list1$. If we write $backup=list1$ and then change $list1$, $backup$ changes too. That's because Python points the name $backup$ to whatever $list1$ points to and treats the two names as synonyms. This can be checked by using the Python query **is**, which is stronger than just checking for equal values.

In [29]:
list1 = [[1, 2, 3], [4, 5, 6]]
backup = list1
print('When list1[0][0] is changed to',list1[0][0],', backup[0][0] changes to',backup[0][0])
backup[1][2]=20
print('When list1[1][2] is changed to',list1[1][2],', backup[1][2] changes to',backup[1][2])
print('Do all the elements of list1 and backup match in value?',list1==backup)
print('Do list1 and backup point to the same object?',list1 is backup)

When list1[0][0] is changed to 1 , backup[0][0] changes to 1
When list1[1][2] is changed to 20 , backup[1][2] changes to 20
Do all the elements of list1 and backup match in value? True
Do list1 and backup point to the same object? True


The safest way to make a full copy is to import the package **copy** and use the command **copy.deepcopy()**.

In [25]:
import copy
list1 = [[1, 2, 3], [4, 5, 6]]
backup = copy.deepcopy(list1)
print('Do all the elements of list1 and backup match in value?',list1==backup)
print('Do list1 and backup point to the same object?',list1 is backup)
list1[0][0] = backup[1][2] = 'X'
print('Index  list1  backup')
print('[0][0]   ',list1[0][0],'    ',backup[0][0])
print('[1][2]   ',list1[1][2],'    ',backup[1][2])

Do all the elements of list1 and backup match in value? True
Do list1 and backup point to the same object? False
Index  list1  backup
[0][0]    X      1
[1][2]    6      X


Tuples can be assigned multiple names too, but because no tuple can be modified without being completely replaced, we don't have to worry that changing the values of one name changes the values of the other.

In [1]:
tuple1 = (1,2,3)   
tuple2 = tuple1 
print('  tuple1   tuple2')
print(tuple1, tuple2)
tuple1 = (4,5,6)
print(tuple1, tuple2)
tuple2 = (7,8,9)
print(tuple1, tuple2)

  tuple1   tuple2
(1, 2, 3) (1, 2, 3)
(4, 5, 6) (1, 2, 3)
(4, 5, 6) (7, 8, 9)
